# Pull data

In [1]:
from passwords import *
from pprint import pprint
import pandas as pd
import pyodbc
import os
import boto3

## Functions

In [2]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name='dustin-payment-analysis'):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

## Constants

In [3]:
# project
str_project = os.getcwd().split('\\')[4].replace('_','-')
print(f'Project: {str_project}')
# output
str_dirname_output = './output'

Project: 20231010-gen-xii


## Output directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

## Read query

In [5]:
str_filepath = './sql/query.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

('with tblMin as\n'
 '(\n'
 'select\n'
 '\tbigDebtorId,\n'
 '\tmin(bigLNRiskViewScoreId) as bigLNRiskViewScoreId\n'
 'from tblLNRiskViewScore\n'
 'group by bigDebtorId\n'
 ')\n'
 ' \n'
 'select\n'
 '\ttblLNRiskViewScore.bigAccountId,\n'
 '\ttblMin.bigDebtorId,\n'
 '\ttblLNRiskViewScore.dtmStampCreation,\n'
 '\ttblLNRiskViewScore.intScore\n'
 'from tblMin\n'
 'left outer join tblLNRiskViewScore on '
 'tblLNRiskViewScore.bigLNRiskViewScoreId=tblMin.bigLNRiskViewScoreId')


## Write into df

In [6]:
%%time

# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)
df = pd.read_sql_query(
    str_query, 
    con=conn,
)
# close
conn.close()
# info
print(f'Data contains {df.shape[0]} rows and {df.shape[1]} columns')

Data contains 2612651 rows and 4 columns
Wall time: 13.4 s


In [7]:
# preview
df.head()

,bigAccountId,bigDebtorId,dtmStampCreation,intScore
0,911457,1209752,2012-04-25 20:17:55.983,549
1,911457,1209751,2012-04-25 20:17:56.030,539
2,911508,1209819,2012-04-26 09:25:55.873,595
3,911706,1210082,2012-04-26 13:22:43.080,511
4,912004,1210461,2012-04-27 07:31:05.983,600


In [8]:
for col in df.columns:
    print(col)

bigAccountId
bigDebtorId
dtmStampCreation
intScore


In [9]:
# get prop nan
df.isnull().mean()

bigAccountId        0.0
bigDebtorId         0.0
dtmStampCreation    0.0
intScore            0.0
dtype: float64

### Lower column names and add suffix

In [10]:
%%time

# lower columns
df.columns = [f'{col.lower()}__ln' for col in df.columns]

# show
df

Wall time: 0 ns


,bigaccountid__ln,bigdebtorid__ln,dtmstampcreation__ln,intscore__ln
0,911457,1209752,2012-04-25 20:17:55.983,549
1,911457,1209751,2012-04-25 20:17:56.030,539
2,911508,1209819,2012-04-26 09:25:55.873,595
3,911706,1210082,2012-04-26 13:22:43.080,511
4,912004,1210461,2012-04-27 07:31:05.983,600
...,...,...,...,...
2612646,5190598,6565834,2020-07-08 15:50:21.897,501
2612647,5190606,6565844,2020-07-08 15:52:46.557,589
2612648,5190607,6565845,2020-07-08 15:52:54.590,577
2612649,5190618,6565859,2020-07-08 15:58:01.520,501


### Save

In [11]:
%%time

# save
str_filename = 'df_riskview_pd.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_csv(str_local_path, index=False)

Wall time: 47.8 s


## Upload to s3

In [12]:
# upload
upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=f'ad_hoc/riskview_score_pd/{str_filename}', 
    str_bucket_name=str_project,
)

## Clean-up

In [13]:
os.remove(str_local_path)